In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GlobalAveragePooling2D, Dropout, Concatenate
)
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
import os

# Load handcrafted features
feature_data = pd.read_csv("../Dataset/Features/signature_features.csv")

# Extract labels, image paths, and handcrafted features
labels = feature_data["label"].values  # 0 = Genuine, 1 = Forged
image_paths = [f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}"
               for file, label in zip(feature_data["filename"], labels)]
handcrafted_features = feature_data.iloc[:, 2:].values  # Exclude filename & label columns

# Split dataset into training and validation sets
train_img_paths, val_img_paths, train_labels, val_labels, train_features, val_features = train_test_split(
    image_paths, labels, handcrafted_features, test_size=0.2, random_state=42, stratify=labels)

# Define image size for InceptionV3
image_size = (299, 299)
input_shape = (299, 299, 3)
feature_dim = handcrafted_features.shape[1]  # Number of handcrafted features


# 🔹 Custom Data Generator for Image + Handcrafted Features
class SignatureDataGenerator(Sequence):
    def __init__(self, image_paths, labels, handcrafted_features, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.labels = labels
        self.handcrafted_features = handcrafted_features
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = np.arange(index * self.batch_size, (index + 1) * self.batch_size)
        batch_image_paths = [self.image_paths[i] for i in batch_indices]
        batch_labels = np.array([self.labels[i] for i in batch_indices])
        batch_handcrafted_features = np.array([self.handcrafted_features[i] for i in batch_indices])

        batch_images = np.array([self.load_image(img_path) for img_path in batch_image_paths])

        return (batch_images, batch_handcrafted_features), batch_labels

    def on_epoch_end(self):
        if self.shuffle:
            indices = np.arange(len(self.image_paths))
            np.random.shuffle(indices)
            self.image_paths = [self.image_paths[i] for i in indices]
            self.labels = self.labels[indices]
            self.handcrafted_features = self.handcrafted_features[indices]

    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)


# Create data generators
batch_size = 32
train_generator = SignatureDataGenerator(train_img_paths, train_labels, train_features, batch_size=batch_size)
val_generator = SignatureDataGenerator(val_img_paths, val_labels, val_features, batch_size=batch_size)


# 🔹 CNN-Dense Hybrid Model
def build_hybrid_model(input_shape, feature_dim):
    # CNN Feature Extractor (InceptionV3)
    base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    cnn_out = Model(base_model.input, x, name="CNN_Feature_Extractor")(base_model.input)

    # Handcrafted Feature Input
    handcrafted_input = Input(shape=(feature_dim,), name="Handcrafted_Input")
    handcrafted_dense = Dense(64, activation="relu")(handcrafted_input)
    handcrafted_dense = Dropout(0.3)(handcrafted_dense)

    # Combine CNN and Handcrafted Features
    merged_features = Concatenate()([cnn_out, handcrafted_dense])

    # Fully Connected Layers (Instead of LSTM)
    dense_out = Dense(64, activation="relu")(merged_features)
    dense_out = Dropout(0.3)(dense_out)
    
    # Final Classification Layer
    output = Dense(1, activation="sigmoid")(dense_out)

    # Build Model
    model = Model(inputs=[base_model.input, handcrafted_input], outputs=output, name="Hybrid_CNN_Dense_Model")

    return model


# Build and Compile Model
model = build_hybrid_model(input_shape, feature_dim)
model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.0001), metrics=["accuracy"])

# 🔹 Train the Model
model.fit(train_generator, validation_data=val_generator, epochs=25)

# Save the Model
model.save("../Model/signature_hybrid_model.h5")

print("Hybrid CNN-LSTM Model training completed and saved!")
